# Workshop 3: Solutions

| Minutes | Activity |
|---|---|
| 0–2 | Getting started |
| 2–10 | Part 1: Solving versus training |
| 10–30 | Part 2: What is a neural network? |
| 30–35 | Part 3: How a network learns |
| 35–38 | Part 4: Reading a network |
| 38–39 | Lab check |
| 39–44 | Demo: all 8 inputs |

## Getting started (2 minutes)

- Save your own copy in Colab (**File → Save a copy in Drive**).
- Click the **play button** beside a code cell to run it. Work from top to bottom.
- Double-click text to edit an answer. Click outside when done.

The first cell loads the libraries from Workshops 1 and 2, then the [California house prices from Workshop 1](https://warwick.ai/education/courses/ml-ai-flagship/workshop-1#getting-started-2-minutes) into a table called `housing`.

This session introduces a new library, **PyTorch**, which provides the building blocks of a neural network.

In [ ]:
# Libraries from Workshops 1 and 2.
import pandas as pd  # Stores data in tables.
import numpy as np  # Does maths on many numbers at once.
import matplotlib.pyplot as plt  # Draws graphs.
from sklearn.datasets import fetch_california_housing  # The house prices from Workshop 1.
from sklearn.linear_model import LinearRegression  # The straight line from Workshop 1.
from sklearn.model_selection import train_test_split  # Splits the data, as in Workshop 1.
from sklearn.preprocessing import StandardScaler  # Rescales inputs, as in Workshop 2.

# New this week: PyTorch.
import torch  # Tensors and automatic gradients.
from torch import nn  # Building blocks for neural networks.

housing = fetch_california_housing(as_frame=True).frame  # Download the data as a table.
housing.head()  # Show the first five rows.

## Part 1: Solving versus training (8 minutes)

In [Workshop 1](https://warwick.ai/education/courses/ml-ai-flagship/workshop-1#2-fit-a-model-and-predict-5-minutes) you fitted a straight line:

$$
\text{value} = \text{slope} \times \text{income} + \text{intercept}
$$

Run the cell to see it again. Expect a slope of about **0.42** and an intercept of about **0.45**.

In [ ]:
line = LinearRegression()
line.fit(housing[["MedInc"]], housing["MedHouseVal"])
print("slope:", line.coef_[0], "| intercept:", line.intercept_)

scikit-learn can find the best line in one go because a straight line is a special case. Its two parameters each just get multiplied by the data, so the best values come from solving two simple equations.

In a neural network, thousands of parameters get multiplied by each other, layer after layer, and equations like that cannot be rearranged into an answer. So instead of solving for a network's parameters, we **train** them. This means starting from random parameters and improving them in small steps, thousands of times. PyTorch gives you the model and the tools, and leaves the rest to you.

The cell below is our own `fit`. It takes any PyTorch model and improves it by repeating four steps: predict, measure the error, work out which way to nudge every parameter, nudge. There is a lot going on here. Run it for now, and we'll explain what's happening later!

In [ ]:
def fit(model, inputs, targets):
    loss_fn = nn.MSELoss()  # The loss from Workshop 1.
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # Makes the improvements.
    for step in range(1000):
        optimizer.zero_grad()  # Forget the last step.
        predicted = model(inputs)  # Predict a value for every house.
        loss = loss_fn(predicted, targets)  # Measure how wrong the predictions are.
        loss.backward()  # Work out which way to nudge every parameter.
        optimizer.step()  # Nudge them.
    return loss.item()

We can build the same model in PyTorch, because linear regression is the simplest possible neural network. `nn.Linear(1, 1)` has one input and one output, and like `LinearRegression` it has a slope and an intercept, which PyTorch calls `weight` and `bias`. `incomes` and `values` are the two columns in PyTorch's format. The cell below prints the slope and intercept before and after training.

In [ ]:
incomes = torch.tensor(housing[["MedInc"]].to_numpy(), dtype=torch.float32)
values = torch.tensor(housing[["MedHouseVal"]].to_numpy(), dtype=torch.float32)

torch.manual_seed(0)
line = nn.Linear(1, 1)
print("before | slope:", line.weight.item(), "| intercept:", line.bias.item())
fit(line, incomes, values)
print("after  | slope:", line.weight.item(), "| intercept:", line.bias.item())

Side by side:

| | scikit-learn | PyTorch |
|---|---|---|
| Model | `LinearRegression()` | `nn.Linear(1, 1)` |
| Fitting | `line.fit(...)` solves | `fit(line, ...)` trains |
| Parameters | `coef_`, `intercept_` | `weight`, `bias` |

The one difference that matters is that scikit-learn's `fit` only works for its own models, and ours works for any PyTorch model. Swap in `nn.Linear(5, 10)`, which takes 5 inputs and gives 10 outputs, and the same `fit` trains it. We now have a recipe for training any neural network we can imagine!

## Part 2: What is a neural network? (20 minutes)

So far the only thing we have trained is a straight line. This part turns it into a neural network, one piece at a time. Each step has a diagram to play with and shows how to write the same thing in PyTorch.

Run each diagram cell and use its controls. The code that draws each diagram is hidden: click **Show code** to see it.

### 1. A neuron is the line plus ReLU

Part 1's model was `nn.Linear(1, 1)`, which is $\text{weight} \times x + \text{bias}$. Pass the result through an **activation function** and you have a **neuron**. There are several to choose from. We use **ReLU**, the most common one. It turns every negative value into 0 and leaves positive values alone, so the output is flat on one side of the point where the line crosses 0, and a straight line on the other, like a hinge.

$$
\text{output} = \text{ReLU}(\text{weight} \times x + \text{bias}), \qquad \text{ReLU}(z) = \max(0, z)
$$

Drag the sliders. The weight sets how steep the line is and which way it faces, and the bias moves the bend left and right.

In [ ]:
# @title One neuron
from ipywidgets import FloatSlider, interact


@interact(weight=FloatSlider(min=-2, max=2, step=0.1, value=1), bias=FloatSlider(min=-3, max=3, step=0.1, value=0))
def one_neuron(weight, bias):
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))

    left.set(xlim=(0, 10), ylim=(0, 4), title="output = ReLU(weight × x + bias)")
    left.axis("off")
    left.scatter([1.2, 5.2], [2, 2], s=[1500, 4000], color=["white", "tab:blue"], edgecolors="black", zorder=2)
    left.text(1.2, 2, "x", ha="center", va="center", fontsize=16)
    left.text(5.2, 2, f"{'+' if bias >= 0 else '−'} {abs(bias):.1f}\nReLU", ha="center", va="center", fontsize=12, color="white")
    left.annotate("", xy=(4.1, 2), xytext=(1.9, 2), arrowprops={"arrowstyle": "->", "linewidth": 2})
    left.text(3, 2.25, f"× {weight:.1f}".replace("-", "−"), ha="center", fontsize=12)
    left.annotate("", xy=(8.4, 2), xytext=(6.3, 2), arrowprops={"arrowstyle": "->", "linewidth": 2})
    left.text(8.5, 2, "output", va="center", fontsize=12)

    x = np.linspace(-5, 5, 201)
    line = weight * x + bias
    right.plot(x, line, "--", color="grey", label="weight × x + bias")
    right.plot(x, np.maximum(line, 0), color="tab:blue", linewidth=3, label="after ReLU")
    if weight != 0 and abs(bias / weight) <= 5:
        right.scatter([-bias / weight], [0], color="black", zorder=3, label="bend")
    right.axhline(0, color="black", linewidth=0.5)
    right.set(xlabel="x (input)", ylabel="output", ylim=(-5, 5))
    right.legend(loc="upper left")
    plt.show()

In PyTorch this is `nn.Sequential(nn.Linear(1, 1), nn.ReLU())`: `nn.Sequential` runs its pieces in order, so the line's output goes into `nn.ReLU()`. Write the same neuron by hand using `torch.relu`, with a weight of 2 and a bias of −4, in one line. Expect `tensor([0., 0., 4., 8., 12., 16.])`.

In [ ]:
x = torch.tensor([0.0, 2.0, 4.0, 6.0, 8.0, 10.0])
torch.relu(2 * x - 4)  # One neuron: weight 2, bias -4.

Where does this neuron bend?

At $x = 2$, where $2x - 4 = 0$. Below 2 the neuron outputs 0. Above 2 its output grows by 2 for every 1 that $x$ grows, because its weight is 2.

### 2. Several neurons from the same input

One neuron gives one bend. `nn.Linear(1, 3)` makes three lines from the same $x$, each with its own weight and bias, and `nn.ReLU()` bends each of them. That is three neurons side by side, a **hidden layer**, and three outputs with nowhere to go yet.

In PyTorch, `nn.Sequential(nn.Linear(1, 3), nn.ReLU())`.

### 3. Combine them with another linear layer

`nn.Linear(3, 1)` takes the three outputs and makes one number from them. It multiplies each by its own **output weight**, adds them together, and adds a bias. That is the same kind of layer as Part 1's line, with three inputs instead of one.

The three neurons below are already set to fit the dots. Add them one at a time and watch the line change. Each coloured dot on the line marks the bend made by the neuron of the same colour.

In [ ]:
# @title Neurons add up
from IPython.display import Markdown, display
from ipywidgets import Checkbox, ToggleButtons, interact

dots_x = np.arange(0, 10.5, 0.5)
dots_y = 1 + np.maximum(dots_x - 2, 0) - 2 * np.maximum(dots_x - 5, 0) + np.maximum(dots_x - 8, 0)
neurons = [(1, -2, 1), (1, -5, -2), (1, -8, 1)]  # (weight, bias, output weight) for each neuron.
output_bias = 1
colours = ["tab:blue", "tab:orange", "tab:green"]
changes = ["starts to climb", "turns down", "flattens out"]
choices = [("No neurons", 0), ("1 neuron", 1), ("2 neurons", 2), ("3 neurons", 3)]


@interact(added=ToggleButtons(options=choices, description=""), relu=Checkbox(value=True, description="ReLU"))
def neurons_add_up(added, relu):
    def output(x):
        total = np.full(len(x), float(output_bias))
        for weight, bias, output_weight in neurons[:added]:
            line = weight * x + bias
            total = total + output_weight * (np.maximum(line, 0) if relu else line)
        return total

    if added == 0:
        display(Markdown(f"No neurons yet: only the output's bias, a flat line at ${output_bias}$."))
    else:
        weight, bias, output_weight = neurons[added - 1]
        if relu:
            display(Markdown(rf"Neuron {added} adds ${output_weight} \times \text{{ReLU}}({weight} \times x - {-bias})$: the line {changes[added - 1]} at $x = {-bias}$."))
        else:
            display(Markdown(rf"Neuron {added} adds ${output_weight} \times ({weight} \times x - {-bias})$: a straight line, so the output stays straight."))

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))

    left.set(xlim=(0, 10), ylim=(-0.5, 4.5), title="Input → hidden layer → output")
    left.axis("off")
    for n, ((weight, bias, output_weight), colour) in enumerate(zip(neurons, colours)):
        y = 3.5 - 1.5 * n
        alpha = 1 if n < added else 0.15
        label = f"ReLU({weight} × x − {-bias})" if relu else f"{weight} × x − {-bias}"
        left.plot([1, 3.6], [2, y], color="grey", alpha=alpha, zorder=1)
        left.plot([6.4, 9], [y, 2], color=colour, linewidth=1 + abs(output_weight), alpha=alpha, zorder=1)
        left.text(5, y, label, ha="center", va="center", fontsize=10, color="white", alpha=alpha, zorder=2,
                  bbox={"boxstyle": "round,pad=0.5", "facecolor": colour, "edgecolor": "black", "alpha": alpha})
        left.text(7.6, (y + 2) / 2 + 0.3, f"× {output_weight}".replace("-", "−"), ha="center", fontsize=10, alpha=alpha)
    left.scatter([1, 9], [2, 2], s=1600, color="white", edgecolors="black", zorder=2)
    left.text(1, 2, "x", ha="center", va="center", fontsize=14)
    left.text(9, 2, f"+ {output_bias}", ha="center", va="center", fontsize=10)

    x = np.linspace(0, 10, 201)
    right.scatter(dots_x, dots_y, color="grey", s=15, zorder=2)
    right.plot(x, output(x), color="black", linewidth=3, zorder=3)
    if relu:
        for (weight, bias, output_weight), colour in zip(neurons[:added], colours):
            bend = -bias / weight
            right.scatter([bend], output(np.array([bend])), s=120, color=colour, edgecolors="black", zorder=4)
    right.set(xlabel="x (input)", xlim=(0, 10), ylim=(-1, 6), title="Output")
    plt.show()

1. What does each neuron add to the line?
2. Untick **ReLU** and add the neurons again. What happens?

1. Each neuron adds one bend, at the point where its $\text{weight} \times x + \text{bias}$ is 0: at $x = 2$, $5$ and $8$. Its output weight decides how much the line's slope changes there: $+1$ starts the climb, $-2$ turns it into a fall, and $+1$ levels it off. With all three, the line goes through every dot. Training finds weights and biases like these for you.
2. Without ReLU, each neuron is just a straight line, so adding one only tilts the output and never bends it. Straight lines always add up to a straight line, however many you use.

This is why a network needs an activation function. Without one, a network is a single straight line however many neurons and layers it has, so making it bigger changes nothing. The activation function gives each neuron its bend, and the bends are what let a network follow curves.

The whole diagram fits in one line of code: the output's bias, plus each neuron times its output weight. Write it and check that you get the dots: `tensor([1., 1., 1., 2., 3., 4., 3., 2., 1., 1., 1.])`.

In [ ]:
x = torch.arange(0.0, 11.0)  # 0, 1, 2, ..., 10
1 + torch.relu(x - 2) - 2 * torch.relu(x - 5) + torch.relu(x - 8)  # The output's bias, plus each neuron times its output weight.

In PyTorch, the whole diagram is `nn.Sequential(nn.Linear(1, 3), nn.ReLU(), nn.Linear(3, 1))`, and its sliders are that model's 10 parameters: 3 weights and 3 biases in the first layer, 3 weights and 1 bias in the second.

So `nn.Linear(a, b)` takes $a$ numbers in and gives $b$ numbers out. Each output is its own weighted sum of the inputs, plus its own bias. That is $a \times b$ weights and $b$ biases.

### 4. Any shape

Now the numbers can be anything. A layer can have as many neurons as you like, and a network can have more than one hidden layer. Each neuron in a layer takes every output of the layer before it. Every line in the diagram below is a weight, and every neuron has a bias. Together, the weights and biases are the network's **parameters**, the numbers that training adjusts.

Drag the sliders and watch the count grow. GPT-3 has 175 billion parameters, and it was trained with the same kind of loop as `fit`.

In [ ]:
# @title Layers and parameters
from ipywidgets import IntSlider, interact


@interact(inputs=IntSlider(min=1, max=8, value=1), neurons=IntSlider(min=1, max=10, value=3), hidden_layers=IntSlider(min=1, max=3, value=1))
def layers_and_parameters(inputs, neurons, hidden_layers):
    sizes = [inputs] + [neurons] * hidden_layers + [1]
    parameters = sum(before * after + after for before, after in zip(sizes, sizes[1:]))  # Weights, then biases.
    names = ["inputs"] + [f"hidden layer {n + 1}" for n in range(hidden_layers)] + ["output"]
    layers = [[(2 * column, (size - 1) / 2 - row) for row in range(size)] for column, size in enumerate(sizes)]

    fig, ax = plt.subplots(figsize=(11, 4.5))
    for before, after in zip(layers, layers[1:]):
        for x1, y1 in before:
            for x2, y2 in after:
                ax.plot([x1, x2], [y1, y2], color="grey", linewidth=0.7, zorder=1)
    for column, (layer, name) in enumerate(zip(layers, names)):
        colour = "tab:blue" if 0 < column < len(layers) - 1 else "white"
        ax.scatter([x for x, y in layer], [y for x, y in layer], s=250, color=colour, edgecolors="black", zorder=2)
        ax.text(2 * column, -5.6, name, ha="center")
    ax.set(xlim=(-1, 2 * len(sizes) - 1), ylim=(-6, 5), title=f"{parameters:,} parameters")
    ax.axis("off")
    plt.show()

PyTorch can count them for you. `example_network.parameters()` gives every weight and bias, and `p.numel()` counts the numbers in each one. Count the parameters of a network with 8 inputs and two hidden layers of 64 neurons. Expect **4801**.

In [ ]:
example_network = nn.Sequential(nn.Linear(8, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))
sum(p.numel() for p in example_network.parameters())  # Add up p.numel() for every p in example_network.parameters().

### 5. Build one for real

House values go up and down with latitude, from San Diego in the south (33°) to the Oregon border (42°), so a straight line cannot follow them. The cell below rescales latitude, as in [Workshop 2](https://warwick.ai/education/courses/ml-ai-flagship/workshop-2#2-see-overfitting), and defines `draw`, which plots a model's predictions over the houses. Run it.

In [ ]:
latitude_scaler = StandardScaler()
latitudes = torch.tensor(latitude_scaler.fit_transform(housing[["Latitude"]].to_numpy()), dtype=torch.float32)
sample = housing.sample(2000, random_state=0)  # Plot 2,000 houses so the graph draws quickly.


def draw(model):
    grid = np.linspace(32.5, 42, 400).reshape(-1, 1)  # 400 latitudes from south to north.
    with torch.no_grad():  # Only predicting, not training.
        curve = model(torch.tensor(latitude_scaler.transform(grid), dtype=torch.float32))
    plt.scatter(sample["Latitude"], sample["MedHouseVal"], s=3, alpha=0.3)
    plt.plot(grid, curve, color="black")
    plt.xlabel("Latitude (degrees north)")
    plt.ylabel("Median house value ($100,000s)")
    plt.show()

Now the network from step 3 with 50 neurons instead of 3, trained on latitude with `fit` from Part 1. Expect a loss of about **1.11**. A straight line gets 1.30.

In [ ]:
model = nn.Sequential(
    nn.Linear(1, 50),  # 50 neurons, each with its own weight and bias.
    nn.ReLU(),  # Bend each one.
    nn.Linear(50, 1),  # Combine them into one prediction.
)
print("loss:", fit(model, latitudes, values))
draw(model)

Los Angeles is at about 34° and San Francisco at about 37.8°. What has the model found?

Two peaks, Los Angeles and the San Francisco Bay Area. Between them is a dip for the cheaper Central Valley, and values fall again further north. Nobody told the model where the cities are, or where to bend. It found both from the data.

Before you run it, predict: what happens if you delete the `nn.ReLU()` line? Then delete it and run the cell again.

A straight line, with a loss of 1.30, the same as a single `nn.Linear(1, 1)`. It is the untick from step 3 on real data. Two linear layers with nothing between them add up to one line, however many neurons are in the middle, so the 151 parameters do no more than 2.

In Workshops 1 and 2 you made lines bend by adding powers such as $x^2$ and $x^3$. You chose the powers, and the powers decided the shape. A network learns its shape instead. Training moves every weight and bias until the bends fit the data, and Part 3 shows how.

## Part 3: How a network learns (5 minutes)

A network starts with random weights and biases. The loss tells you how wrong they are, but not which way to change each one.

To see how training decides, take a single parameter: the slope of the income line from Part 1, with the intercept fixed at 0.45. Each slope gives a different loss, and plotted together they make a **bowl**. The best slope, 0.42, sits at the bottom.

Click **Step** to take one step downhill, and keep clicking. Then click **Reset**, set the learning rate to 0.05 and try again: the dot overshoots the bottom and zigzags before it settles. At 0.06 it overshoots further every time.

In [ ]:
# @title Loss bowl
from IPython.display import display
from ipywidgets import Button, FloatSlider, HBox, Layout, Output, VBox

a, b, c = 18.59, -15.54, 3.95  # The loss is a × slope² + b × slope + c, with the intercept fixed at 0.45.
path = [0.0]
rate = FloatSlider(min=0.005, max=0.06, step=0.005, value=0.01, description="learning rate", readout_format=".3f",
                   style={"description_width": "initial"}, layout=Layout(width="350px"))
step_button = Button(description="Step")
reset_button = Button(description="Reset")
picture = Output()


def loss(slope):
    return a * slope ** 2 + b * slope + c


def gradient(slope):
    return 2 * a * slope + b


def draw(*_):
    slope = path[-1]
    following = slope - rate.value * gradient(slope)
    with picture:
        picture.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7, 3.8))
        slopes = np.linspace(-0.6, 1.4, 201)
        ax.plot(slopes, loss(slopes), color="grey")
        ax.scatter(path[:-1], [loss(s) for s in path[:-1]], color="tab:blue", alpha=0.35, s=20)
        if -0.6 <= slope <= 1.4:
            ax.annotate("", xy=(following, loss(following)), xytext=(slope, loss(slope)),
                        arrowprops={"arrowstyle": "->", "color": "tab:orange", "linewidth": 2})
            ax.scatter([slope], [loss(slope)], color="tab:blue", edgecolors="black", s=80, zorder=3)
            title = f"Step {len(path) - 1}: slope {slope:.2f}, loss {loss(slope):.2f}, gradient {gradient(slope):.1f}".replace("-", "−")
        else:
            title = "The dot has left the bowl: each step overshoots by more than the last."
        ax.set(xlim=(-0.6, 1.4), ylim=(0, 20), xlabel="slope (intercept fixed at 0.45)", ylabel="loss (MSE)", title=title)
        plt.show()


def step(_):
    if -0.6 <= path[-1] <= 1.4:
        path.append(path[-1] - rate.value * gradient(path[-1]))
        draw()


def reset(_):
    path[:] = [0.0]
    draw()


step_button.on_click(step)
reset_button.on_click(reset)
rate.observe(draw, names="value")
display(VBox([HBox([step_button, reset_button, rate]), picture]))
draw()

Each step changes the slope by the learning rate times the gradient:

$$
\text{slope} \leftarrow \text{slope} - \text{learning rate} \times \text{gradient}
$$

- The **gradient** says how steep the bowl is under the dot, and which way is uphill. The step goes the other way, which is why the formula subtracts it. The orange arrow shows the step.
- `loss.backward()` works out the gradient for every parameter at once, however many there are.
- The **optimiser** turns each gradient into a step. `SGD` multiplies it by the learning rate. **Adam** also adapts the size of each parameter's steps as it goes.

Training a network, as `fit` does, is this picture repeated thousands of times, for every weight and bias at once.

Now find a gradient yourself. The cell below makes a line with a random slope and intercept, and measures its loss on the house prices. Add the backward pass. Expect a slope gradient of about **−15.2**.

In [ ]:
torch.manual_seed(0)
line = nn.Linear(1, 1)  # A straight line with a random slope and intercept.
loss = nn.MSELoss()(line(incomes), values)  # How wrong it is.
loss.backward()  # Find the gradient for every parameter.
print("slope:", line.weight.item(), "| slope gradient:", line.weight.grad.item())

The gradient is negative. Will the next step make the slope bigger or smaller?

Bigger. The step subtracts the learning rate times the gradient, and subtracting a negative number adds. It is the same as the first arrow in the bowl, which points right.

The cell below is the loop inside `fit`, using plain gradient descent (`SGD`) so that the learning rate is the only setting. Run it, then change `learning_rate` to 0.1 and run it again.

In [ ]:
learning_rate = 0.01  # Then try 0.1.

torch.manual_seed(0)
line = nn.Linear(1, 1)
optimizer = torch.optim.SGD(line.parameters(), lr=learning_rate)
for step in range(101):
    optimizer.zero_grad()
    loss = nn.MSELoss()(line(incomes), values)
    loss.backward()
    optimizer.step()
    if step % 20 == 0:
        print("step", step, "| loss", loss.item())

What happens to the loss with a learning rate of 0.1? Why?

At 0.01 the loss falls to about 0.70. At 0.1 it grows with every step, reaching about $7 \times 10^{18}$ by step 20, then `inf` and finally `nan` (not a number). Each step overshoots the bottom of the bowl by more than the last, like 0.06 in the diagram.

## Part 4: Reading a network (3 minutes)

Here is the network we build in the demo: `example_network` from step 4 of Part 2, with 8 inputs and two hidden layers of 64 neurons. Run the cell. PyTorch lists the layers in the order they run, each one's output feeding the next.

In [ ]:
nn.Sequential(
    nn.Linear(8, 64),  # 64 neurons, each with 8 weights, one per input, and a bias.
    nn.ReLU(),  # Bend every one of them.
    nn.Linear(64, 64),  # A second hidden layer. Each neuron takes all 64 outputs of the first.
    nn.ReLU(),
    nn.Linear(64, 1),  # One weighted sum of the last 64 neurons, plus a bias: the prediction.
)

Every `nn.Linear(a, b)` is $b$ neurons with $a$ weights and a bias each, and every `nn.ReLU()` bends the neurons before it. There is no bend after the last layer, because its output is the prediction itself. Altogether that is the 4801 parameters you counted in step 4, and `fit` moves every one of them with the step from Part 3, a thousand times over.

## Lab check (1 minute)

Explain in one sentence each:

- Why a network needs ReLU between its layers.
- What one step of `fit` does.

- Without ReLU, every layer is a straight line, and lines added up are still a line, so extra layers add nothing. ReLU adds the bends.
- It predicts a value for every house, measures the loss, works out the gradient for every parameter with `loss.backward()`, and moves each parameter a small step downhill.

## Demo: all 8 inputs (5 minutes)

Now the task from Workshop 1: predict house values from all 8 inputs. The cell below makes [Workshop 1's split](https://warwick.ai/education/courses/ml-ai-flagship/workshop-1#t1-split-the-data-three-ways-3-minutes), so the scores compare, and rescales the inputs as in Workshop 2. Run it.

In [ ]:
train, rest = train_test_split(housing, test_size=0.3, random_state=0)  # The split from Workshop 1.
val, test = train_test_split(rest, test_size=0.5, random_state=0)  # The test set stays aside.
input_columns = housing.columns.drop("MedHouseVal")  # The 8 input columns.

scaler = StandardScaler()  # Learns each column's average and spread from the training set only.
X_train = torch.tensor(scaler.fit_transform(train[input_columns]), dtype=torch.float32)
X_val = torch.tensor(scaler.transform(val[input_columns]), dtype=torch.float32)
y_train = torch.tensor(train[["MedHouseVal"]].to_numpy(), dtype=torch.float32)
y_val = torch.tensor(val[["MedHouseVal"]].to_numpy(), dtype=torch.float32)

Then the network from Part 4, trained with `fit` and scored on the validation set. We build it live: fill in the five layers. Expect a validation MSE of about **0.29**.

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(8, 64),  # 8 inputs into 64 neurons.
    nn.ReLU(),  # Bend them.
    nn.Linear(64, 64),  # A second hidden layer of 64 neurons.
    nn.ReLU(),  # Bend them.
    nn.Linear(64, 1),  # One prediction.
)
print("training MSE:", fit(model, X_train, y_train))
with torch.no_grad():  # Only measuring, not training.
    print("validation MSE:", nn.MSELoss()(model(X_val), y_val).item())

Compare it with [Workshop 1's models](https://warwick.ai/education/courses/ml-ai-flagship/workshop-1#take-home-find-the-sweet-spot-about-15-minutes), all scored on the same validation set. (Workshop 1's 0.52 for linear regression was scored on the data it trained on.)

| Model | Validation MSE |
|---|---|
| Always predict the average | 1.33 |
| Linear regression, 8 inputs | 0.54 |
| Decision tree, depth 9 | 0.43 |
| Random forest | 0.28 |
| This network | 0.29 |

Five lines of PyTorch, trained with the same `fit` as the straight line in Part 1, come within 0.01 of the random forest. Other random starts land anywhere between about 0.28 and 0.34, which is why the cell sets a seed.